# CHARTA Step 34 — Fine-tune ClinicalBERT with LoRA on BC5CDR (CID Relation Extraction)

This notebook fine-tunes `emilyalsentzer/Bio_ClinicalBERT` for Chemical-Induced Disease (CID)
relation extraction using LoRA on the BC5CDR dataset. Designed for **Google Colab T4 GPU**.

**All code is self-contained in this notebook — no CHARTA source upload needed.**

**Workflow:**
1. Mount Google Drive (optional — for weight backup)
2. Install dependencies
3. Verify GPU & create directories
4. Load BC5CDR dataset & build entity-pair examples
5. Load ClinicalBERT + apply LoRA + tokenize
6. Fine-tune (~2 hours on T4)
7. Verify & test saved weights
8. Download weights to local machine

After downloading, place weights in `CHARTA/models/lora_weights/clinicalbert_rel/` locally.

In [ ]:
#@title Step 1 — Mount Google Drive (optional, for backup)  {display-mode: "form"}
# Mounting Drive is optional but recommended — if the Colab session dies during
# the 2-hour training, your weights will still be saved in Drive.
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/CHARTA', exist_ok=True)
print("Drive mounted at /content/drive/MyDrive/CHARTA")

In [ ]:
#@title Step 2 — Install dependencies  {display-mode: "form"}
# Colab has PyTorch pre-installed — do NOT reinstall it (breaks CUDA drivers)
!pip install -q transformers==4.40.0 peft==0.10.0 accelerate==0.29.3 \
                "datasets>=2.14.0" scikit-learn
print("Dependencies installed")

In [ ]:
#@title Step 3 — Verify GPU & create directories  {display-mode: "form"}
import torch, os

assert torch.cuda.is_available(), "No GPU! Go to Runtime > Change runtime type > T4 GPU"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

os.makedirs("models/lora_weights/clinicalbert_rel", exist_ok=True)
os.makedirs("logs", exist_ok=True)
print("Output directories created")

In [ ]:
#@title Step 4 — Load BC5CDR dataset & build entity-pair examples  {display-mode: "form"}
from datasets import load_dataset, Dataset

def _get_first(val):
    """Extract first element if list, else return as-is. Handles BigBio format variations."""
    if isinstance(val, list):
        return val[0] if val else ""
    return val if val else ""

def _build_entity_pair_examples(dataset_split):
    """Convert BC5CDR KB split to (text, label) classification rows for CID relation extraction."""
    examples = []
    for item in dataset_split:
        # Collect all entities across passages in this document
        all_diseases = []
        all_chemicals = []
        passage_texts = []
        for passage in item.get("passages", []):
            text = _get_first(passage.get("text", ""))
            if text:
                passage_texts.append(text)
            for e in passage.get("entities", []):
                if e.get("type") == "Disease":
                    all_diseases.append(e)
                elif e.get("type") == "Chemical":
                    all_chemicals.append(e)

        full_text = " ".join(passage_texts)

        # Build relation set from document-level relations
        rels = set()
        for r in item.get("relations", []):
            arg1 = _get_first(r.get("arg1_id", ""))
            arg2 = _get_first(r.get("arg2_id", ""))
            rels.add((arg1, arg2))

        # Create entity-pair classification examples
        for d in all_diseases:
            for c in all_chemicals:
                c_id = _get_first(c.get("id", ""))
                d_id = _get_first(d.get("id", ""))
                c_text = _get_first(c.get("text", ""))
                d_text = _get_first(d.get("text", ""))
                label = 1 if (c_id, d_id) in rels or (d_id, c_id) in rels else 0
                inp = f"[E1] {c_text} [/E1] {full_text} [E2] {d_text} [/E2]"
                examples.append({"text": inp, "label": label})
    return examples

# Load BC5CDR dataset (bigbio_kb config — the ONLY valid config for this format)
# trust_remote_code=True is required for datasets >= 2.16.0 with custom loading scripts
try:
    ds = load_dataset("bigbio/bc5cdr", "bc5cdr_bigbio_kb", trust_remote_code=True)
except TypeError:
    # Older datasets version does not support trust_remote_code parameter
    ds = load_dataset("bigbio/bc5cdr", "bc5cdr_bigbio_kb")

# Handle split name variations (validation vs dev)
train_split = ds["train"]
if "validation" in ds:
    val_split = ds["validation"]
elif "dev" in ds:
    val_split = ds["dev"]
else:
    val_split = ds["test"]  # fallback (not ideal but prevents crash)

print(f"Train split: {len(train_split)} documents")
print(f"Validation split: {len(val_split)} documents")

train_rows = _build_entity_pair_examples(train_split)
val_rows = _build_entity_pair_examples(val_split)
print(f"Train entity-pair examples: {len(train_rows)}")
print(f"Validation entity-pair examples: {len(val_rows)}")

# Show label distribution
train_pos = sum(r["label"] for r in train_rows)
val_pos = sum(r["label"] for r in val_rows)
print(f"Train CID positives: {train_pos}/{len(train_rows)} ({train_pos/len(train_rows)*100:.1f}%)")
print(f"Val CID positives: {val_pos}/{len(val_rows)} ({val_pos/len(val_rows)*100:.1f}%)")
print("Dataset loaded & entity-pair examples built")

In [ ]:
#@title Step 5 — Load ClinicalBERT + apply LoRA + tokenize datasets  {display-mode: "form"}
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset

MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"
OUTPUT_DIR = "models/lora_weights/clinicalbert_rel/"
LORA_RANK = 8
LORA_ALPHA = 16
LORA_TARGET_MODULES = ["query", "value"]

# Load base model & tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

# Apply LoRA — only to BERT attention layers (query, value)
# Do NOT apply LoRA to GraphSAGE or other non-attention models
lora_cfg = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    target_modules=LORA_TARGET_MODULES,
    lora_dropout=0.1,
    bias="none",
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

# Tokenize datasets
def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=256)

train_ds = Dataset.from_list(train_rows).map(tokenize, batched=True)
val_ds = Dataset.from_list(val_rows).map(tokenize, batched=True)
train_ds.set_format("torch", columns=["input_ids", "attention_mask", "label"])
val_ds.set_format("torch", columns=["input_ids", "attention_mask", "label"])

print(f"Train dataset: {len(train_ds)} examples")
print(f"Val dataset: {len(val_ds)} examples")
print("Model loaded, LoRA applied, datasets tokenized")

In [ ]:
#@title Step 6 — Fine-tune ClinicalBERT with LoRA (~2 hours on T4)  {display-mode: "form"}
from transformers import TrainingArguments, Trainer

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=10,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    learning_rate=2e-4,
    weight_decay=0.01,
    fp16=True,  # requires GPU — OK on Colab T4
    logging_steps=50,
    report_to="none",
)

trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds)
print("Starting LoRA fine-tuning of ClinicalBERT ...")
trainer.train()
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"LoRA weights saved to {OUTPUT_DIR}")

In [ ]:
#@title Step 7 — Verify saved LoRA weights & backup to Drive  {display-mode: "form"}
import os, shutil

weights_dir = "models/lora_weights/clinicalbert_rel/"
files_saved = os.listdir(weights_dir)
print("Files saved:", sorted(files_saved))

# Check required files
assert "adapter_config.json" in files_saved, "adapter_config.json missing!"
has_weights = any(f in files_saved for f in ["adapter_model.safetensors", "adapter_model.bin"])
assert has_weights, "No adapter weights file found!"
print("  adapter_config.json found")
print("  adapter weights file found (PEFT 0.10.0 saves .safetensors by default)")

# Print file sizes
for f in sorted(files_saved):
    path = os.path.join(weights_dir, f)
    if os.path.isfile(path):
        size_kb = os.path.getsize(path) / 1024
        print(f"  {f}: {size_kb:.1f} KB")

# Backup to Google Drive (if mounted) — critical to prevent data loss if session dies
drive_backup = "/content/drive/MyDrive/CHARTA/clinicalbert_rel_lora_backup"
if os.path.isdir("/content/drive/MyDrive/CHARTA"):
    shutil.copytree(weights_dir, drive_backup, dirs_exist_ok=True)
    print(f"Backup copied to Drive: {drive_backup}")
else:
    print("Drive not mounted — skip backup (weights only in Colab session)")

print("All LoRA weight files verified")

In [ ]:
#@title Step 8 — Quick inference test  {display-mode: "form"}
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import PeftModel
import torch

MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"
OUTPUT_DIR = "models/lora_weights/clinicalbert_rel/"

# Load base model + LoRA adapter
tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR)
base_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)
model.eval()

# Test with a sample CID sentence
test_input = "[E1] aspirin [/E1] The patient was treated with aspirin for [E2] myocardial infarction [/E2]"
inputs = tokenizer(test_input, return_tensors="pt", truncation=True, max_length=256)

with torch.no_grad():
    logits = model(**inputs).logits
    probs = torch.softmax(logits, dim=-1)
    label = torch.argmax(logits, dim=-1).item()

print(f"Input     : {test_input}")
print(f"Label     : {label}  (0=No relation, 1=CID)")
print(f"Confidence: {probs[0][label].item():.4f}")
print("Inference test passed")

In [ ]:
#@title Step 9 — Download LoRA weights as zip  {display-mode: "form"}
import zipfile, os, shutil
from google.colab import files

zip_path = "/content/clinicalbert_rel_lora.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for fname in os.listdir("models/lora_weights/clinicalbert_rel/"):
        fpath = f"models/lora_weights/clinicalbert_rel/{fname}"
        if os.path.isfile(fpath):
            z.write(fpath, fname)

print(f"Zip created: {zip_path} ({os.path.getsize(zip_path) / 1e6:.1f} MB)")

# Option A: download directly to browser
files.download(zip_path)

# Option B: also copy to Drive for persistent storage
drive_zip = "/content/drive/MyDrive/CHARTA/clinicalbert_rel_lora.zip"
if os.path.isdir("/content/drive/MyDrive/CHARTA"):
    shutil.copy(zip_path, drive_zip)
    print(f"Also saved to Drive: {drive_zip}")
else:
    print("Drive not mounted — weights only available via browser download")

print("Download started — extract into CHARTA/models/lora_weights/clinicalbert_rel/ locally")

## Post-Download: Local Setup Instructions

After downloading and extracting the zip into `CHARTA/models/lora_weights/clinicalbert_rel/`:

1. **Verify locally:**
   ```bash
   cd CHARTA/src
   python -c "from layer2.relation_extractor import load_relation_model; t, m = load_relation_model(); print('LoRA model loaded OK')"
   ```

2. **Run Layer 2 pipeline with relations:**
   ```bash
   python run_layer2.py --mode run
   ```

3. **The relation extractor will now return CID relations instead of empty lists.**

**Local folder structure after extraction:**
```
CHARTA/
  models/
    lora_weights/
      clinicalbert_rel/
        adapter_config.json
        adapter_model.safetensors
        tokenizer.json
        tokenizer_config.json
        vocab.txt
        special_tokens_map.json
```

## Optional: Evaluate NER F1 on NCBI Disease

This step is **optional** and requires installing scispaCy. It evaluates the scispaCy NER model
on the NCBI Disease test split. Expected: micro-F1 > 0.75.

This is independent of the LoRA fine-tuning above — skip if you only need the LoRA weights.

In [ ]:
#@title Step 10 — (Optional) Evaluate NER F1  {display-mode: "form"}
# This requires scispaCy — install first (adds ~2 min)
!pip install -q scispacy
!pip install -q https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_ner_bc5cdr_md-0.5.4.tar.gz

from datasets import load_dataset
from sklearn.metrics import f1_score
import spacy

SCISPACY_MODEL = "en_ner_bc5cdr_md"
NCBI_DISEASE_DATASET = "ncbi/ncbi_disease"

nlp = spacy.load(SCISPACY_MODEL)
try:
    ds = load_dataset(NCBI_DISEASE_DATASET, trust_remote_code=True, split="test")
except TypeError:
    # Older datasets version does not support trust_remote_code
    ds = load_dataset(NCBI_DISEASE_DATASET, split="test")

all_true, all_pred = [], []
for example in ds:
    tokens = example["tokens"]
    gold = example["ner_tags"]
    text = " ".join(tokens)
    doc = nlp(text)
    pred_bio = ["O"] * len(tokens)
    for ent in doc.ents:
        start_tok = len(text[:ent.start_char].split())
        end_tok = len(text[:ent.end_char].split())
        for j in range(start_tok, min(end_tok, len(tokens))):
            pred_bio[j] = "B-DISEASE" if j == start_tok else "I-DISEASE"
    gold_bio = [ds.features["ner_tags"].feature.int2str(g) for g in gold]
    all_true.extend(gold_bio)
    all_pred.extend(pred_bio)

f1 = f1_score(all_true, all_pred, average="micro",
              labels=[l for l in set(all_true) if l != "O"])
print(f"NER micro-F1: {f1:.4f}  (target > 0.75)")